# Optimización computacional

Cada mejora se sustenta primero en la **complejidad asintótica** (en función de $n$ filas, $p$ variables, $k$ vecinos, $T$ árboles de profundidad $d$ y $m$ componentes) y después se contrasta con el **tiempo empírico**. Las mediciones se hicieron con el experimento principal terminado (sin competencia por los núcleos), con la mediana de 3 repeticiones.

In [1]:
import os  # Rutas
import sys  # Configuración del intérprete
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count()))  # Evita un aviso de joblib en Windows
sys.path[:0] = [os.path.abspath(".."), os.path.abspath("../.."), os.path.abspath("../../entregable1")]  # e3, src y e1
import warnings  # Control de avisos
warnings.filterwarnings("ignore")  # Oculta avisos de versiones
import contextlib, io  # Silenciar un diagnóstico de joblib en Windows
from joblib.externals.loky import cpu_count  # Conteo de núcleos
with contextlib.redirect_stderr(io.StringIO()):  # joblib imprime un diagnóstico al no encontrar «wmic»
    cpu_count(only_physical_cores=True)  # Se calcula una vez (queda en caché)
import pickle  # Lectura de resultados
import numpy as np  # Cálculo numérico
import pandas as pd  # Tablas
import plotly.express as px  # Figuras rápidas
import plotly.graph_objects as go  # Figuras detalladas
from plotly.subplots import make_subplots  # Paneles
from IPython.display import Markdown, display  # Texto enriquecido
from e1.figuras import mostrar, AZUL, ROJO, VERDE, AMBAR, GRIS, MORADO  # Estilo de figuras (tema claro del informe)
from e3 import config as c3, datos as dd, modelos as mo, optimizadores as op, experimento as ex  # Código del entregable
c3.fijar_semilla()  # Semilla global (42)
pd.set_option("display.width", 170)  # Ancho de las tablas
pd.set_option("display.max_columns", 30)  # Columnas visibles
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")  # Cuatro decimales
COLORES = {"Grid Search": GRIS, "Random Search": AMBAR, "Bayesiana (Optuna TPE)": AZUL, "Genética (DEAP)": VERDE, "Successive Halving": MORADO}  # Color por optimizador
import joblib  # Lectura
C = joblib.load(c3.RESULTADOS / "complejidad.joblib")  # Resultados del análisis computacional
C["teoria"]  # Complejidad teórica

,entrenamiento,inferencia
KNN (fuerza bruta),O(1) (almacena),O(n·p) por consulta
KNN (KD-Tree / Ball Tree),O(p·n log n),≈ O(log n) por consulta en baja dimensión; se ...
"KNN (FAISS HNSW, aproximado)",O(n log n),"≈ O(log n) por consulta, con pérdida controlad..."
Naive Bayes,O(n·p),O(p) por fila
Regresión logística,O(n·p) por iteración,O(p) por fila
Árbol de decisión,O(p·n log n),O(d) por fila
Random Forest,"O(T·p'·n log n), p' = √p",O(T·d) por fila
XGBoost (hist),O(T·n·p) (histogramas de B contenedores),O(T·d) por fila
XGBoost (exact),O(T·p·n log n) (ordena cada variable),O(T·d) por fila
SVM núcleo RBF exacto,O(n²·p) a O(n³),O(n_sv·p) por fila


## 1. Complejidad teórica frente a tiempo empírico

Se mide el tiempo de ajuste y de predicción variando $n$ de 2 500 a 80 000 filas. Si el tiempo crece como $n^{b}$, la pendiente de $\log t$ frente a $\log n$ estima el **exponente empírico** $b$, que se compara con el teórico.

In [2]:
tiempos, exponentes = C["escalamiento"]  # Tiempos y exponentes
fig = make_subplots(1, 2, subplot_titles=("Ajuste", "Predicción de 1 000 filas"))  # Dos paneles
for j, (nombre, g) in enumerate(tiempos.groupby("modelo")):  # Cada modelo
    col = px.colors.qualitative.Bold[j % 10]  # Color
    fig.add_scatter(x=g["n"], y=g["ajuste (s)"], mode="lines+markers", name=nombre, line=dict(color=col), row=1, col=1)  # Ajuste
    fig.add_scatter(x=g["n"], y=g["predicción 1000 filas (s)"], mode="lines+markers", showlegend=False, line=dict(color=col), row=1, col=2)  # Predicción
fig.update_xaxes(type="log", title_text="n (log)")  # Escala log
fig.update_yaxes(type="log", title_text="segundos (log)")  # Escala log
fig.update_layout(title="Escalamiento empírico (ejes logarítmicos: la pendiente es el exponente)")  # Título
mostrar(fig, 460)  # Muestra
teorico = {"KNN": "0 (ajuste) / 1 (predicción)", "Naive Bayes": "1 / 0", "Regresión logística": "≈ 1 / 0", "Árbol de decisión": "≈ 1 (n log n) / 0",
           "Random Forest": "≈ 1 (n log n) / 0", "XGBoost": "≈ 1 / 0", "SVM (Nyström + lineal)": "1 / 0", "SVM RBF exacto": "2 a 3 / ≈ 1 (n_sv crece con n)"}
exponentes.assign(**{"exponente teórico (ajuste / predicción)": pd.Series(teorico)})

,exponente ajuste,exponente predicción,exponente teórico (ajuste / predicción)
KNN,0.9669,0.6839,0 (ajuste) / 1 (predicción)
Naive Bayes,0.7503,-0.2724,1 / 0
Random Forest,1.1101,0.1197,≈ 1 (n log n) / 0
Regresión logística,0.9996,-0.0137,≈ 1 / 0
SVM (Nyström + lineal),1.1334,0.0403,1 / 0
SVM RBF exacto,1.9873,0.8670,2 a 3 / ≈ 1 (n_sv crece con n)
XGBoost,0.6213,-0.0126,≈ 1 / 0
Árbol de decisión,1.1395,0.1959,≈ 1 (n log n) / 0


**Interpretación.** Los exponentes empíricos coinciden con la teoría en los casos claros: la regresión logística escala linealmente (1,00); el árbol y Random Forest, como $n \log n$ (1,11–1,14); la SVM con Nyström, linealmente (1,13); y la **SVM con núcleo exacto, cuadráticamente (1,99)**, con una predicción que también crece con $n$ (0,87) porque crece el número de vectores de soporte. Las desviaciones tienen explicación: KNN no «solo almacena» (scikit-learn construye un árbol en el ajuste, de ahí el exponente ≈ 1, y la búsqueda en árbol hace la predicción sublineal, 0,68); Naive Bayes y XGBoost (0,62–0,75) quedan por debajo de 1 porque en este rango de $n$ pesan los costos fijos (en XGBoost, construir los histogramas de contenedores). En los modelos de costo constante por fila, el tiempo de predicción no depende de $n$ (exponente ≈ 0).

## 2. KNN: búsqueda exhaustiva, KD-Tree, Ball Tree y FAISS

La búsqueda exhaustiva cuesta $O(n \cdot p)$ por consulta. KD-Tree y Ball Tree podan regiones del espacio ($\approx O(\log n)$ en baja dimensión) pero con $p = 34$ la poda pierde eficacia (maldición de la dimensionalidad). FAISS HNSW construye un grafo navegable de varias capas y encuentra vecinos **aproximados** en tiempo casi logarítmico; el costo es una fracción de vecinos incorrectos (recall@k < 1).

In [3]:
C["knn"]

,construcción (s),consulta 1000 (s),recall@k frente a exacto,AUC-ROC (voto de vecinos)
método,,,,
scikit-learn brute,0.0110,0.0432,1.0000,0.8895
scikit-learn kd_tree,0.2648,2.5269,1.0000,0.8895
scikit-learn ball_tree,0.1363,3.5497,1.0000,0.8895
FAISS exacto (IndexFlatL2),0.0242,0.0359,1.0000,0.8895
"FAISS aproximado (HNSW, M = 32)",0.3155,0.0033,0.9321,0.8882


**Interpretación.** Con $p = 34$, KD-Tree y Ball Tree son **60–80 veces más lentos** que la fuerza bruta (2,5–3,5 s frente a 0,04 s por 1 000 consultas): la poda deja de funcionar en dimensión media (maldición de la dimensionalidad), mientras que la fuerza bruta se reduce a un producto de matrices optimizado (BLAS). FAISS **HNSW** responde 13 veces más rápido que la fuerza bruta (0,003 s) recuperando el 93 % de los vecinos exactos, con un AUC-ROC casi idéntico (0,888 frente a 0,890): el compromiso exactitud–velocidad es muy favorable porque el voto de 51 vecinos tolera que algunos sean aproximados.

## 3. Modelos lineales: solvers (SAGA)

SAGA es un gradiente estocástico con reducción de varianza: cada iteración cuesta $O(p)$ por fila ($O(n \cdot p)$ por época), admite L1 y escala a datos grandes y dispersos. Se compara con lbfgs/liblinear (logística) y con Cholesky / descenso por coordenadas (Ridge / Lasso), con las 132 728 filas del entrenamiento.

In [4]:
C["lineales"]

,ajuste (s),métrica en 2022,métrica
modelo,,,
Regresión logística (lbfgs),0.3783,0.8978,AUC-ROC
Regresión logística (liblinear),0.6448,0.8981,AUC-ROC
Regresión logística (saga),2.4813,0.8978,AUC-ROC
Ridge (auto),0.0537,0.4610,RMSE
Ridge (saga),9.7345,0.4610,RMSE
Lasso (descenso por coordenadas),9.1334,0.4611,RMSE
"Lasso (SGD, penalización L1)",0.2640,0.4633,RMSE


**Interpretación.** Con datos densos, estandarizados y solo 34 variables, **SAGA no es la opción más rápida**: lbfgs ajusta la logística en 0,4 s frente a 2,5 s de SAGA, y Ridge con su solución cerrada (Cholesky, $O(np^2 + p^3)$) tarda 0,05 s frente a 9,7 s, con métricas idénticas. SAGA brilla cuando $n$ y $p$ son enormes o dispersos, o con L1 (es el único solver de la logística que admite todas las penalizaciones). Para Lasso, el SGD con L1 es 35 veces más rápido que el descenso por coordenadas con un RMSE casi igual (0,463 frente a 0,461). La lección: el solver eficiente depende de la forma de los datos, y aquí se verifica en lugar de suponerse.

## 4. Naive Bayes incremental (`partial_fit`)

Naive Bayes gaussiano solo necesita, por clase, conteos, medias y varianzas: se actualizan por lotes con las fórmulas de Chan et al., así que el modelo final es el mismo y los datos nunca necesitan estar completos en memoria.

In [5]:
C["naive_bayes"]

,AUC-ROC 2022,memoria pico (MB),máx. diferencia de medias
fit completo,0.8981,64.8526,0.0000
partial_fit en 10 lotes,0.8981,6.7107,0.0000


**Interpretación.** `partial_fit` en 10 lotes produce **exactamente el mismo modelo** (diferencia de medias 0, mismo AUC) con **10 veces menos memoria** pico (6,7 MB frente a 64,9 MB): los estadísticos suficientes de Naive Bayes se actualizan exactamente. Permite entrenar con datos que no caben en memoria o que llegan semana a semana.

## 5. XGBoost: `hist` frente a `exact` y parada temprana

In [6]:
C["xgboost"]

,ajuste (s),árboles,AUC-PR 2022
variante,,,
tree_method = exact (300 árboles),23.2525,300,0.7088
tree_method = hist (300 árboles),1.4618,300,0.7078
"hist + parada temprana (máx. 2 000, paciencia 50)",1.5170,125,0.7120


**Interpretación.** El método `hist` es **16 veces más rápido** que `exact` (1,5 s frente a 23,3 s) con el mismo AUC-PR: discretiza cada variable en contenedores y evita ordenar las 132 728 filas en cada división. La **parada temprana** con 2022 como conjunto independiente se detiene en 125 árboles (de un máximo de 2 000) y mejora el AUC-PR (0,712 frente a 0,708 con 300 árboles): menos cómputo y menos sobreajuste a la vez.

## 6. SVM: núcleo exacto frente a aproximaciones

In [7]:
svm = C["svm"]  # Tiempos por variante y tamaño
fig = make_subplots(1, 2, subplot_titles=("Ajuste (s)", "AUC-PR en 2022"))  # Dos paneles
for j, (v, g) in enumerate(svm.groupby("variante")):  # Variantes
    col = [ROJO, AZUL, VERDE, AMBAR][j % 4]  # Color
    fig.add_scatter(x=g["n"], y=g["ajuste (s)"], mode="lines+markers", name=v, line=dict(color=col), row=1, col=1)  # Tiempo
    fig.add_scatter(x=g["n"], y=g["AUC-PR 2022"], mode="lines+markers", showlegend=False, line=dict(color=col), row=1, col=2)  # Calidad
fig.update_xaxes(type="log", title_text="n (log)")  # Escala
fig.update_yaxes(type="log", row=1, col=1)  # Escala
mostrar(fig, 400)  # Muestra
svm.pivot_table(index="variante", columns="n", values=["ajuste (s)", "AUC-PR 2022"]).round(3)

AUC-PR 2022                      ajuste (s)                     
n                               2500   5000   10000  20000      2500   5000   10000  20000
variante                                                                                  
LinearSVC                      0.6720 0.6690 0.6690 0.6720     0.0080 0.0160 0.0260 0.0750
Nyström (300) + LinearSVC      0.6540 0.6580 0.6610 0.6730     0.0820 0.1360 0.2490 0.5720
SGD (bisagra)                  0.4900 0.6090 0.6570 0.6570     0.0070 0.0170 0.0310 0.0440
SVC RBF exacto                 0.6130 0.6410 0.6090 0.6260     0.0470 0.1640 0.8130 3.1900

**Interpretación.** El tiempo del núcleo exacto crece de forma **cuadrática**: de 0,05 s a 3,2 s al pasar de 2 500 a 20 000 filas (×64 para ×8 filas), y su AUC-PR (0,61–0,64 con hiperparámetros por defecto) no mejora con más datos. Nyström + LinearSVC escala linealmente (0,57 s con 20 000 filas) y logra AUC-PR 0,673; LinearSVC sin núcleo es aún más rápido (0,08 s) con un AUC-PR casi igual, señal de que el problema es casi lineal en estas variables. SGD necesita más datos para converger (0,49 con 2 500 filas, 0,66 desde 10 000). Por eso la SVM del experimento usa Nyström: conserva la flexibilidad del núcleo RBF sin el costo cuadrático.

## 7. Paralelización

Dos niveles: **entre combinaciones** (el experimento corrió 10 procesos con `joblib`/`loky`, cada uno con `n_jobs = 1` para no sobresuscribir los núcleos) y **dentro de un modelo** (`n_jobs` de Random Forest).

In [8]:
C["paralelo"]

,segundos,aceleración,eficiencia
configuración,,,
"Random Forest, n_jobs = 1",17.2830,1.0000,1.0000
"Random Forest, n_jobs = 2",9.1001,1.8992,0.9496
"Random Forest, n_jobs = 4",5.1835,3.3343,0.8336
"Random Forest, n_jobs = 8",3.5186,4.9119,0.6140
"4 pliegues de regresión logística, backend loky",1.2183,NaN,NaN
"4 pliegues de regresión logística, backend threading",0.2278,NaN,NaN
"4 pliegues de regresión logística, secuencial",0.3096,NaN,NaN


**Interpretación.** Random Forest acelera 1,9× con 2 núcleos, 3,3× con 4 y 4,9× con 8 (eficiencia 0,61): la paralelización de árboles es casi perfecta hasta 4 núcleos, y después pesan las partes secuenciales (ley de Amdahl) y que el equipo tiene 6 núcleos físicos (12 lógicos). Para tareas cortas, paralelizar con procesos (**loky**) es **más lento** que no paralelizar (1,22 s frente a 0,31 s): crear procesos y serializar los datos cuesta más que el cálculo; con **hilos** (0,23 s) sí se gana, porque los solvers liberan el GIL. Por eso el experimento paraleliza al nivel más grueso, **entre combinaciones** (10 procesos con `n_jobs = 1` dentro de cada modelo): 420 tareas en 105 minutos. **Dask o Ray** solo se justificarían para repartir las tareas entre varias máquinas; en un solo equipo de 12 núcleos, joblib basta. **GPU:** el equipo no tiene GPU NVIDIA, así que `device='cuda'` de XGBoost no pudo medirse; con `hist` en CPU el ajuste ya tarda ≈ 1,5 s.

## 8. Perfilamiento y memoria

In [9]:
C["perfil"].head(15)

,función,llamadas,tiempo propio (s),tiempo acumulado (s)
0,ejecutar_tarea (ielSR/Projects/proyecto_dengue...,1,0.0004,10.3826
1,fit (/imblearn/pipeline.py),9,0.0004,7.7336
2,inner_f (/xgboost/core.py),108,0.0010,7.0816
3,fit (/xgboost/sklearn.py),9,0.0005,7.0806
4,bayesiana (lSR/Projects/proyecto_dengue/entreg...,1,0.0000,6.9334
5,optimize (/optuna/study/study.py),1,0.0000,6.9330
6,_optimize (/optuna/study/_optimize.py),1,0.0000,6.9330
7,_optimize_sequential (/optuna/study/_optimize.py),1,0.0001,6.9330
8,_run_trial (/optuna/study/_optimize.py),4,0.0001,6.9329
9,f (lSR/Projects/proyecto_dengue/entregable3/e3...,4,0.0001,6.9305


In [10]:
C["memoria"]

,memoria adicional pico (MB)
KNN,19.7344
Naive Bayes,18.6562
Regresión logística,10.3906
Árbol de decisión,5.1914
Random Forest,12.1250
XGBoost (hist),7.2734
SVM Nyström + lineal,279.2930


**Interpretación.** El perfil de una tarea anidada muestra que el **≈ 70 % del tiempo** está en el entrenamiento de XGBoost (`update` de los árboles) y ≈ 23 % en cargar los datos, un costo que se paga una sola vez por proceso gracias a la caché (`lru_cache`). Optuna, el remuestreo y el Pipeline cuestan casi nada: optimizarlos sería prematuro. En memoria, la SVM con Nyström es la más exigente (≈ 280 MB: la matriz de 300 componentes por fila), mientras los árboles necesitan pocos MB.

## 9. Tabla comparativa: modelo estándar frente a optimizado

In [11]:
knn, lin, xg, nb = C["knn"], C["lineales"], C["xgboost"], C["naive_bayes"]  # Resultados
sv = C["svm"][C["svm"]["n"] == C["svm"]["n"].max()].set_index("variante")  # SVM con el n más grande
mem = C["memoria"]["memoria adicional pico (MB)"]  # Memoria
comparacion = pd.DataFrame([
    {"modelo": "KNN", "estándar": "fuerza bruta", "optimizado": "FAISS HNSW", "complejidad estándar → optimizada (consulta)": "O(n·p) → ≈ O(log n)",
     "consulta 1000 estándar (s)": knn.loc["scikit-learn brute", "consulta 1000 (s)"], "consulta 1000 optimizada (s)": knn.loc["FAISS aproximado (HNSW, M = 32)", "consulta 1000 (s)"],
     "Δ desempeño": knn.loc["FAISS aproximado (HNSW, M = 32)", "AUC-ROC (voto de vecinos)"] - knn.loc["scikit-learn brute", "AUC-ROC (voto de vecinos)"]},
    {"modelo": "XGBoost", "estándar": "exact, 300 árboles", "optimizado": "hist + parada temprana", "complejidad estándar → optimizada (consulta)": "O(T·p·n log n) → O(T'·n·p), T' < T",
     "ajuste estándar (s)": xg.iloc[0]["ajuste (s)"], "ajuste optimizado (s)": xg.iloc[2]["ajuste (s)"], "Δ desempeño": xg.iloc[2]["AUC-PR 2022"] - xg.iloc[0]["AUC-PR 2022"]},
    {"modelo": "SVM", "estándar": "núcleo RBF exacto", "optimizado": "Nyström + LinearSVC", "complejidad estándar → optimizada (consulta)": "O(n²·p)–O(n³) → O(n·m·p)",
     "ajuste estándar (s)": sv.loc["SVC RBF exacto", "ajuste (s)"], "ajuste optimizado (s)": sv.loc["Nyström (300) + LinearSVC", "ajuste (s)"],
     "Δ desempeño": sv.loc["Nyström (300) + LinearSVC", "AUC-PR 2022"] - sv.loc["SVC RBF exacto", "AUC-PR 2022"]},
    {"modelo": "Regresión logística", "estándar": "lbfgs", "optimizado": "saga", "complejidad estándar → optimizada (consulta)": "O(n·p) por iteración (ambos)",
     "ajuste estándar (s)": lin.loc["Regresión logística (lbfgs)", "ajuste (s)"], "ajuste optimizado (s)": lin.loc["Regresión logística (saga)", "ajuste (s)"],
     "Δ desempeño": lin.loc["Regresión logística (saga)", "métrica en 2022"] - lin.loc["Regresión logística (lbfgs)", "métrica en 2022"]},
    {"modelo": "Naive Bayes", "estándar": "fit completo", "optimizado": "partial_fit por lotes", "complejidad estándar → optimizada (consulta)": "O(n·p) (igual); memoria O(n·p) → O(lote·p)",
     "memoria estándar (MB)": nb.iloc[0]["memoria pico (MB)"], "memoria optimizada (MB)": nb.iloc[1]["memoria pico (MB)"], "Δ desempeño": nb.iloc[1]["AUC-ROC 2022"] - nb.iloc[0]["AUC-ROC 2022"]},
]).set_index("modelo")
comparacion

,estándar,optimizado,complejidad estándar → optimizada (consulta),consulta 1000 estándar (s),consulta 1000 optimizada (s),Δ desempeño,ajuste estándar (s),ajuste optimizado (s),memoria estándar (MB),memoria optimizada (MB)
modelo,,,,,,,,,,
KNN,fuerza bruta,FAISS HNSW,O(n·p) → ≈ O(log n),0.0432,0.0033,-0.0013,NaN,NaN,NaN,NaN
XGBoost,"exact, 300 árboles",hist + parada temprana,"O(T·p·n log n) → O(T'·n·p), T' < T",NaN,NaN,0.0032,23.2525,1.5170,NaN,NaN
SVM,núcleo RBF exacto,Nyström + LinearSVC,O(n²·p)–O(n³) → O(n·m·p),NaN,NaN,0.0466,3.1899,0.5716,NaN,NaN
Regresión logística,lbfgs,saga,O(n·p) por iteración (ambos),NaN,NaN,0.0000,0.3783,2.4813,NaN,NaN
Naive Bayes,fit completo,partial_fit por lotes,O(n·p) (igual); memoria O(n·p) → O(lote·p),NaN,NaN,0.0000,NaN,NaN,64.8526,6.7107


**Interpretación.** Las variantes optimizadas logran: KNN **13× más rápido** en las consultas (FAISS HNSW, −0,001 de AUC); XGBoost **15× más rápido** en el ajuste y con mejor AUC-PR (hist + parada temprana); SVM **5,6× más rápida** y con +0,047 de AUC-PR (Nyström frente al núcleo exacto con valores por defecto); Naive Bayes con **10× menos memoria** y el mismo modelo. La excepción documentada es la logística con SAGA, más lenta que lbfgs en este conjunto denso y pequeño en $p$.